# 7. Plot phase and mechanistic figures

Versione pulita: cartelle ordinate in `reports/02_phase_features` e `reports/04_mechanistic_analysis`, con sottocartelle `exploratory_figures` e `paper_figures`.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() or (candidate / ".git").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Project root not found. Run the notebook from inside the project repository.")

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def report_dir(root: Path, analysis_name: str, subfolder: str | None = None) -> Path:
    base = root / "reports" / analysis_name
    return ensure_dir(base if subfolder is None else base / subfolder)

ROOT = find_project_root()
print("ROOT:", ROOT)


ROOT: /home/eleonorab/repository/convective_cooling


In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = find_project_root()

INPUT_FILE = ROOT / "data" / "processed" / "features" / "phase_features_long.csv"
OUTPUT_DIR = report_dir(ROOT, "02_phase_features", "exploratory_figures")
ensure_dir(OUTPUT_DIR)

PHASE_ORDER = [
    "standing_1",
    "walk_low_1",
    "walk_mod_1",
    "walk_low_2",
    "walk_mod_2",
    "standing_2",
]

PHASE_LABELS = {
    "standing_1": "Phase 1",
    "walk_low_1": "Phase 2",
    "walk_mod_1": "Phase 3",
    "walk_low_2": "Phase 4",
    "walk_mod_2": "Phase 5",
    "standing_2": "Phase 6",
}

CONDITION_ORDER = [
    "no_wind",
    "wind_low",
    "wind_high",
]

CONDITION_COLORS = {
    "no_wind": "blue",
    "wind_low": "red",
    "wind_high": "black",
}

SIGNALS_TO_PLOT = [
    "t_dorsal",
    "t_microclimateback",
    "rh_microclimateback",
    "mean_skin_temperature",
    "t_abdomen",
    "t_chest",
    "t_neck",
    "t_hand",
    "t_shin",
    "rh_dorsal",
    "rh_abdomen",
    "rh_chest",
    "rh_neck",
    "rh_hand",
    "rh_shin",
    "t_microclimatefront",
    "rh_microclimatefront",
    "t_environment",
    "rh_environment",
    "t_microclimate_gradient",
    "rh_microclimate_gradient",
]


def load_data() -> pd.DataFrame:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Missing file: {INPUT_FILE}")

    df = pd.read_csv(INPUT_FILE)

    for col in ["mean", "std", "median", "min", "max", "delta_in_out", "slope", "baseline_value", "delta_mean_from_baseline"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "phase" in df.columns:
        df["phase"] = pd.Categorical(df["phase"], categories=PHASE_ORDER, ordered=True)

    if "condition" in df.columns:
        df["condition"] = df["condition"].astype(str).str.strip()
        df["condition"] = pd.Categorical(df["condition"], categories=CONDITION_ORDER, ordered=True)

    if "signal" in df.columns:
        df["signal"] = df["signal"].astype(str).str.strip()

    return df


def get_available_signals(df: pd.DataFrame) -> list[str]:
    available = sorted(df["signal"].dropna().unique().tolist())
    return [s for s in SIGNALS_TO_PLOT if s in available]


def make_phase_delta_plot(df: pd.DataFrame, signal: str) -> None:
    sub = df[df["signal"] == signal].copy()
    sub = sub.dropna(subset=["condition", "phase", "delta_mean_from_baseline"])

    if sub.empty:
        print(f"Skipping {signal}: no valid rows")
        return

    plot_df = (
        sub.groupby(["condition", "phase"], observed=True)["delta_mean_from_baseline"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    if plot_df.empty:
        print(f"Skipping {signal}: grouped dataframe empty")
        return

    plot_df["sem"] = plot_df["std"] / plot_df["count"].pow(0.5)

    plt.figure(figsize=(8, 5))

    for condition in CONDITION_ORDER:
        cond_df = plot_df[plot_df["condition"] == condition].sort_values("phase")
        if cond_df.empty:
            continue

        x = [PHASE_LABELS[p] for p in cond_df["phase"].astype(str).tolist()]
        y = cond_df["mean"].to_numpy(dtype=float)
        sem = cond_df["sem"].fillna(0).to_numpy(dtype=float)

        plt.plot(
        x,
        y,
        marker="o",
        linewidth=2,
        color=CONDITION_COLORS[condition],
        label=condition,
        )

        plt.fill_between(
            range(len(x)),
            y - sem,
            y + sem,
            color=CONDITION_COLORS[condition],
            alpha=0.15,
        )

    plt.axhline(0, linestyle="--", linewidth=1, color="gray", alpha=0.7)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_linewidth(2.5)

    plt.xlabel("Phase")
    plt.ylabel("Δ from baseline")
    plt.legend(frameon=False)

    # show only bottom ticks / labels and rotate x ticks
    ax.tick_params(axis="y", which="both", left=False, labelleft=False)
    ax.tick_params(axis="x", which="both", rotation=45)

    plt.tight_layout()

    out_path = OUTPUT_DIR / f"{signal}_delta_by_phase.png"
    plt.savefig(out_path, dpi=300)
    plt.close()

    print(f"Saved: {out_path}")


def main() -> None:
    df = load_data()
    signals = get_available_signals(df)

    if not signals:
        raise ValueError("No target signals found in phase_features_long.csv")

    print("Signals to plot:")
    for s in signals:
        print(" -", s)

    for signal in signals:
        make_phase_delta_plot(df, signal)

    print(f"\nDone. Plots saved in: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Signals to plot:
 - t_dorsal
 - t_microclimateback
 - rh_microclimateback
 - mean_skin_temperature
 - t_abdomen
 - t_chest
 - t_neck
 - t_hand
 - t_shin
 - rh_dorsal
 - rh_abdomen
 - rh_chest
 - rh_neck
 - rh_hand
 - rh_shin
 - t_microclimatefront
 - rh_microclimatefront
 - t_environment
 - rh_environment
 - rh_microclimate_gradient
Saved: /home/eleonorab/repository/convective_cooling/reports/02_phase_features/exploratory_figures/t_dorsal_delta_by_phase.png
Saved: /home/eleonorab/repository/convective_cooling/reports/02_phase_features/exploratory_figures/t_microclimateback_delta_by_phase.png
Saved: /home/eleonorab/repository/convective_cooling/reports/02_phase_features/exploratory_figures/rh_microclimateback_delta_by_phase.png
Saved: /home/eleonorab/repository/convective_cooling/reports/02_phase_features/exploratory_figures/mean_skin_temperature_delta_by_phase.png
Saved: /home/eleonorab/repository/convective_cooling/reports/02_phase_features/exploratory_figures/t_abdomen_delta_by_phase

In [3]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


ROOT = find_project_root()
INPUT_DIR = ROOT / "data" / "processed" / "aggregated_by_signal"
OUTPUT_DIR = report_dir(ROOT, "04_mechanistic_analysis", "exploratory_figures")
ensure_dir(OUTPUT_DIR)

CONDITION_ORDER = ["no_wind", "wind_low", "wind_high"]
BIN_SECONDS = 10


def load_signal(signal_file: str, value_name: str) -> pd.DataFrame:
    path = INPUT_DIR / signal_file
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_csv(path)

    required_cols = ["subject_id", "condition", "elapsed_seconds", "elapsed_minutes", "value"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path.name}: {missing}")

    df = df[required_cols].copy()
    df["subject_id"] = df["subject_id"].astype(str).str.strip()
    df["condition"] = df["condition"].astype(str).str.strip()
    df["elapsed_seconds"] = pd.to_numeric(df["elapsed_seconds"], errors="coerce")
    df["elapsed_minutes"] = pd.to_numeric(df["elapsed_minutes"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["subject_id", "condition", "elapsed_seconds", "value"]).copy()

    df["elapsed_seconds_bin"] = (df["elapsed_seconds"] / BIN_SECONDS).round() * BIN_SECONDS
    df["elapsed_minutes_bin"] = df["elapsed_seconds_bin"] / 60.0

    df = (
        df.groupby(
            ["subject_id", "condition", "elapsed_seconds_bin", "elapsed_minutes_bin"],
            as_index=False
        )["value"]
        .mean()
    )

    df = df.rename(columns={"value": value_name})
    return df


def build_pair_df(file_x: str, name_x: str, file_y: str, name_y: str) -> pd.DataFrame:
    dx = load_signal(file_x, name_x)
    dy = load_signal(file_y, name_y)

    merged = pd.merge(
        dx,
        dy,
        on=["subject_id", "condition", "elapsed_seconds_bin", "elapsed_minutes_bin"],
        how="inner"
    )

    merged = merged.dropna(subset=[name_x, name_y]).copy()

    print(f"\nPAIR DATASET: {name_y} vs {name_x}")
    print("shape:", merged.shape)
    print("conditions:", merged["condition"].value_counts(dropna=False).to_dict())

    return merged


def print_pair_debug(df: pd.DataFrame, x: str, y: str) -> None:
    print(f"\nDEBUG for {y} vs {x}")
    for cond in CONDITION_ORDER:
        sub = df[df["condition"] == cond].copy()
        print(f"\nCONDITION: {cond}")
        print("rows:", len(sub))
        if len(sub) > 0:
            print(f"{x} non-null:", sub[x].notna().sum())
            print(f"{y} non-null:", sub[y].notna().sum())


def get_local_limits(plot_df: pd.DataFrame, x: str, y: str) -> tuple[tuple[float, float], tuple[float, float]]:
    x_low, x_high = plot_df[x].quantile([0.02, 0.98])
    y_low, y_high = plot_df[y].quantile([0.02, 0.98])

    x_pad = (x_high - x_low) * 0.08 if x_high > x_low else 0.5
    y_pad = (y_high - y_low) * 0.08 if y_high > y_low else 0.5

    return (x_low - x_pad, x_high + x_pad), (y_low - y_pad, y_high + y_pad)


def plot_one_condition(
    df: pd.DataFrame,
    x: str,
    y: str,
    condition: str,
    filename: str,
) -> None:
    plot_df = df[df["condition"] == condition].copy()

    if plot_df.empty:
        print(f"Skipping {filename}: no data for {condition}")
        return

    xlim, ylim = get_local_limits(plot_df, x, y)

    plt.figure(figsize=(7, 6))

    sns.scatterplot(
        data=plot_df,
        x=x,
        y=y,
        alpha=0.3
    )

    sns.regplot(
        data=plot_df,
        x=x,
        y=y,
        scatter=False
    )

    plt.xlim(xlim)
    plt.ylim(ylim)

    plt.xlabel(x.replace("_", " "))
    plt.ylabel(y.replace("_", " "))
    plt.title(f"{condition}: {y} vs {x}")
    plt.tight_layout()

    out_path = OUTPUT_DIR / filename
    plt.savefig(out_path, dpi=300)
    plt.close()

    print("Saved:", out_path)


def main():
    print("ROOT:", ROOT)
    print("INPUT_DIR:", INPUT_DIR)
    print("OUTPUT_DIR:", OUTPUT_DIR)

    # TEMPERATURA
    df_temp = build_pair_df(
        "t_microclimateback.csv", "t_microclimateback",
        "t_dorsal.csv", "t_dorsal"
    )
    print_pair_debug(df_temp, "t_microclimateback", "t_dorsal")

    for condition in CONDITION_ORDER:
        plot_one_condition(
            df_temp,
            x="t_microclimateback",
            y="t_dorsal",
            condition=condition,
            filename=f"{condition}_t_microclimateback_vs_t_dorsal.png",
        )

    # UMIDITA
    df_rh = build_pair_df(
        "rh_microclimateback.csv", "rh_microclimateback",
        "rh_dorsal.csv", "rh_dorsal"
    )
    print_pair_debug(df_rh, "rh_microclimateback", "rh_dorsal")

    for condition in CONDITION_ORDER:
        plot_one_condition(
            df_rh,
            x="rh_microclimateback",
            y="rh_dorsal",
            condition=condition,
            filename=f"{condition}_rh_microclimateback_vs_rh_dorsal.png",
        )

    print(f"\nDone. Mechanistic plots saved in:\n{OUTPUT_DIR}")


if __name__ == "__main__":
    main()

ROOT: /home/eleonorab/repository/convective_cooling
INPUT_DIR: /home/eleonorab/repository/convective_cooling/data/processed/aggregated_by_signal
OUTPUT_DIR: /home/eleonorab/repository/convective_cooling/reports/04_mechanistic_analysis/exploratory_figures

PAIR DATASET: t_dorsal vs t_microclimateback
shape: (7190, 6)
conditions: {'no_wind': 2698, 'wind_high': 2336, 'wind_low': 2156}

DEBUG for t_dorsal vs t_microclimateback

CONDITION: no_wind
rows: 2698
t_microclimateback non-null: 2698
t_dorsal non-null: 2698

CONDITION: wind_low
rows: 2156
t_microclimateback non-null: 2156
t_dorsal non-null: 2156

CONDITION: wind_high
rows: 2336
t_microclimateback non-null: 2336
t_dorsal non-null: 2336
Saved: /home/eleonorab/repository/convective_cooling/reports/04_mechanistic_analysis/exploratory_figures/no_wind_t_microclimateback_vs_t_dorsal.png
Saved: /home/eleonorab/repository/convective_cooling/reports/04_mechanistic_analysis/exploratory_figures/wind_low_t_microclimateback_vs_t_dorsal.png
Saved:

In [4]:
# ============================================================
# Paper-oriented mechanistic plots
# Run this cell after the previous mechanistic-plot cell
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PAPER_MECH_DIR = report_dir(ROOT, "04_mechanistic_analysis", "paper_figures")
ensure_dir(PAPER_MECH_DIR)

CONDITION_ORDER = ["no_wind", "wind_low", "wind_high"]

CONDITION_LABELS = {
    "no_wind": "No ventilation",
    "wind_low": "Low ventilation",
    "wind_high": "High ventilation",
}

PHASE_WINDOWS = [
    (0, 10, "Phase 1"),
    (10, 20, "Phase 2"),
    (20, 30, "Phase 3"),
    (30, 40, "Phase 4"),
    (40, 50, "Phase 5"),
    (50, 60, "Phase 6"),
]

PHASE_LABELS = {
    "Phase 1": "Phase 1",
    "Phase 2": "Phase 2",
    "Phase 3": "Phase 3",
    "Phase 4": "Phase 4",
    "Phase 5": "Phase 5",
    "Phase 6": "Phase 6",
}


def add_phase_column(df):
    df = df.copy()
    df["phase"] = np.nan

    for start, end, phase in PHASE_WINDOWS:
        mask = (df["elapsed_minutes_bin"] >= start) & (df["elapsed_minutes_bin"] < end)
        df.loc[mask, "phase"] = phase

    return df.dropna(subset=["phase"]).copy()


def get_global_limits(df, x, y, q_low=0.02, q_high=0.98):
    x_low, x_high = df[x].quantile([q_low, q_high])
    y_low, y_high = df[y].quantile([q_low, q_high])

    x_pad = (x_high - x_low) * 0.08
    y_pad = (y_high - y_low) * 0.08

    return (x_low - x_pad, x_high + x_pad), (y_low - y_pad, y_high + y_pad)


def plot_hexbin_by_condition(df, x, y, filename, xlabel, ylabel):
    plot_df = df.dropna(subset=[x, y]).copy()
    xlim, ylim = get_global_limits(plot_df, x, y)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)

    hb_ref = None

    for ax, condition in zip(axes, CONDITION_ORDER):
        sub = plot_df[plot_df["condition"] == condition]

        hb = ax.hexbin(
            sub[x],
            sub[y],
            gridsize=35,
            mincnt=1
        )

        hb_ref = hb

        ax.set_title(CONDITION_LABELS.get(condition, condition))
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_xlabel(xlabel)
        ax.spines[["top", "right"]].set_visible(False)

    axes[0].set_ylabel(ylabel)

    if hb_ref is not None:
        cbar = fig.colorbar(hb_ref, ax=axes, shrink=0.85)
        cbar.set_label("Point density")

    #fig.suptitle(f"{ylabel} vs {xlabel}: density distribution", y=1.02)
    #fig.tight_layout()

    out_path = PAPER_MECH_DIR / filename
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", out_path)


def plot_phase_colored_scatter(df, x, y, filename, xlabel, ylabel):
    plot_df = add_phase_column(df)
    plot_df = plot_df.dropna(subset=[x, y, "phase"]).copy()

    xlim, ylim = get_global_limits(plot_df, x, y)

    phase_order = [p[2] for p in PHASE_WINDOWS]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)

    for ax, condition in zip(axes, CONDITION_ORDER):
        sub = plot_df[plot_df["condition"] == condition]

        for phase in phase_order:
            phase_sub = sub[sub["phase"] == phase]

            ax.scatter(
                phase_sub[x],
                phase_sub[y],
                s=14,
                alpha=0.35,
                label=PHASE_LABELS.get(phase, phase)
            )

        ax.set_title(CONDITION_LABELS.get(condition, condition))
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_xlabel(xlabel)
        ax.spines[["top", "right"]].set_visible(False)

    axes[0].set_ylabel(ylabel)

    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="center right",
        bbox_to_anchor=(1.08, 0.5),
        frameon=False
    )

    fig.suptitle(f"{ylabel} vs {xlabel}: phase-dependent states", y=1.02)
    #fig.tight_layout()

    out_path = PAPER_MECH_DIR / filename
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", out_path)


def plot_mean_phase_space_trajectory(df, x, y, filename, xlabel, ylabel):
    plot_df = add_phase_column(df)
    plot_df = plot_df.dropna(subset=[x, y]).copy()

    mean_traj = (
        plot_df
        .groupby(["condition", "elapsed_minutes_bin"], as_index=False)
        .agg(
            x_mean=(x, "mean"),
            y_mean=(y, "mean")
        )
    )

    xlim, ylim = get_global_limits(plot_df, x, y)

    fig, ax = plt.subplots(figsize=(7, 6))

    for condition in CONDITION_ORDER:
        sub = mean_traj[mean_traj["condition"] == condition].sort_values("elapsed_minutes_bin")

        ax.plot(
            sub["x_mean"],
            sub["y_mean"],
            linewidth=2,
            label=CONDITION_LABELS.get(condition, condition)
        )

        ax.scatter(
            sub["x_mean"].iloc[0],
            sub["y_mean"].iloc[0],
            s=60,
            marker="o"
        )

        ax.scatter(
            sub["x_mean"].iloc[-1],
            sub["y_mean"].iloc[-1],
            s=80,
            marker="X"
        )

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f"Mean phase-space trajectory: {ylabel} vs {xlabel}")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    #ax.tight_layout()

    out_path = PAPER_MECH_DIR / filename
    fig.savefig(out_path, dpi=300)
    plt.close(fig)

    print("Saved:", out_path)


# ============================================================
# Rebuild datasets using your existing functions
# ============================================================

df_temp = build_pair_df(
    "t_microclimateback.csv", "t_microclimateback",
    "t_dorsal.csv", "t_dorsal"
)

df_rh = build_pair_df(
    "rh_microclimateback.csv", "rh_microclimateback",
    "rh_dorsal.csv", "rh_dorsal"
)


# ============================================================
# Temperature plots
# ============================================================

plot_hexbin_by_condition(
    df_temp,
    x="t_microclimateback",
    y="t_dorsal",
    filename="paper_hexbin_temperature_microclimate_vs_dorsal.png",
    xlabel=r"Back microclimate temperature [°C]",
    ylabel=r"Dorsal skin temperature [°C]"
)

plot_phase_colored_scatter(
    df_temp,
    x="t_microclimateback",
    y="t_dorsal",
    filename="paper_phase_scatter_temperature_microclimate_vs_dorsal.png",
    xlabel=r"Back microclimate temperature [°C]",
    ylabel=r"Dorsal skin temperature [°C]"
)

plot_mean_phase_space_trajectory(
    df_temp,
    x="t_microclimateback",
    y="t_dorsal",
    filename="paper_trajectory_temperature_microclimate_vs_dorsal.png",
    xlabel=r"Back microclimate temperature [°C]",
    ylabel=r"Dorsal skin temperature [°C]"
)


# ============================================================
# Relative humidity plots
# ============================================================

plot_hexbin_by_condition(
    df_rh,
    x="rh_microclimateback",
    y="rh_dorsal",
    filename="paper_hexbin_rh_microclimate_vs_dorsal.png",
    xlabel=r"Back microclimate relative humidity [%]",
    ylabel=r"Dorsal relative humidity [%]"
)

plot_phase_colored_scatter(
    df_rh,
    x="rh_microclimateback",
    y="rh_dorsal",
    filename="paper_phase_scatter_rh_microclimate_vs_dorsal.png",
    xlabel=r"Back microclimate relative humidity [%]",
    ylabel=r"Dorsal relative humidity [%]"
)

plot_mean_phase_space_trajectory(
    df_rh,
    x="rh_microclimateback",
    y="rh_dorsal",
    filename="paper_trajectory_rh_microclimate_vs_dorsal.png",
    xlabel=r"Back microclimate relative humidity [%]",
    ylabel=r"Dorsal relative humidity [%]"
)

print(f"\nDone. Paper mechanistic plots saved in:\n{PAPER_MECH_DIR}")


PAIR DATASET: t_dorsal vs t_microclimateback
shape: (7190, 6)
conditions: {'no_wind': 2698, 'wind_high': 2336, 'wind_low': 2156}

PAIR DATASET: rh_dorsal vs rh_microclimateback
shape: (7368, 6)
conditions: {'no_wind': 2697, 'wind_high': 2336, 'wind_low': 2335}
Saved: /home/eleonorab/repository/convective_cooling/reports/04_mechanistic_analysis/paper_figures/paper_hexbin_temperature_microclimate_vs_dorsal.png


/tmp/ipykernel_6533/1216106240.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Phase 1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, "phase"] = phase


Saved: /home/eleonorab/repository/convective_cooling/reports/04_mechanistic_analysis/paper_figures/paper_phase_scatter_temperature_microclimate_vs_dorsal.png
Saved: /home/eleonorab/repository/convective_cooling/reports/04_mechanistic_analysis/paper_figures/paper_trajectory_temperature_microclimate_vs_dorsal.png


/tmp/ipykernel_6533/1216106240.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Phase 1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, "phase"] = phase


Saved: /home/eleonorab/repository/convective_cooling/reports/04_mechanistic_analysis/paper_figures/paper_hexbin_rh_microclimate_vs_dorsal.png


/tmp/ipykernel_6533/1216106240.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Phase 1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, "phase"] = phase


Saved: /home/eleonorab/repository/convective_cooling/reports/04_mechanistic_analysis/paper_figures/paper_phase_scatter_rh_microclimate_vs_dorsal.png
Saved: /home/eleonorab/repository/convective_cooling/reports/04_mechanistic_analysis/paper_figures/paper_trajectory_rh_microclimate_vs_dorsal.png

Done. Paper mechanistic plots saved in:
/home/eleonorab/repository/convective_cooling/reports/04_mechanistic_analysis/paper_figures


/tmp/ipykernel_6533/1216106240.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Phase 1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, "phase"] = phase


In [5]:
# ============================================================
# Paper-oriented phase response plots
# Run after the exploratory phase plots cell
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PAPER_PHASE_DIR = report_dir(ROOT, "02_phase_features", "paper_figures")
ensure_dir(PAPER_PHASE_DIR)

CONDITION_LABELS = {
    "no_wind": "No ventilation",
    "wind_low": "Low ventilation",
    "wind_high": "High ventilation",
}

CONDITION_COLORS = {
    "no_wind": "blue",
    "wind_low": "red",
    "wind_high": "black",
}

PHASE_LABELS = {
    "standing_1": "Phase 1",
    "walk_low_1": "Phase 2",
    "walk_mod_1": "Phase 3",
    "walk_low_2": "Phase 4",
    "walk_mod_2": "Phase 5",
    "standing_2": "Phase 6",
}

MAIN_PANEL_SIGNALS = {
    "mean_skin_temperature": r"Mean skin temperature [$^\circ$C]",
    "t_dorsal": r"Dorsal skin temperature [$^\circ$C]",
    "t_microclimateback": r"Back microclimate temperature [$^\circ$C]",
    "rh_microclimateback": r"Back microclimate RH [%]",
}

RH_PANEL_SIGNALS = {
    "rh_dorsal": r"Dorsal RH [%]",
    "rh_chest": r"Chest RH [%]",
    "rh_microclimatefront": r"Front microclimate RH [%]",
    "rh_microclimateback": r"Back microclimate RH [%]",
}


def summarize_phase_delta(df, signals):
    sub = df[
        df["signal"].isin(signals)
    ].dropna(
        subset=["signal", "condition", "phase", "delta_mean_from_baseline"]
    ).copy()

    summary = (
        sub
        .groupby(["signal", "condition", "phase"], observed=True)["delta_mean_from_baseline"]
        .agg(mean="mean", std="std", count="count")
        .reset_index()
    )

    summary["sem"] = summary["std"] / np.sqrt(summary["count"])
    return summary


def plot_signal_panel(summary, signal_map, filename):
    available_signals = [s for s in signal_map if s in summary["signal"].unique()]

    if not available_signals:
        print(f"Skipping {filename}: no available signals")
        return

    n = len(available_signals)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(18, 5 * nrows),
        sharex=True
    )

    axes = np.array(axes).reshape(-1)

    x = np.arange(len(PHASE_ORDER))
    x_labels = [PHASE_LABELS[p] for p in PHASE_ORDER]

    handles = None
    labels = None

    for ax, signal in zip(axes, available_signals):
        sig_df = summary[summary["signal"] == signal].copy()

        for condition in CONDITION_ORDER:
            cond_df = (
                sig_df[sig_df["condition"] == condition]
                .set_index("phase")
                .reindex(PHASE_ORDER)
            )

            if cond_df["mean"].isna().all():
                continue

            y = cond_df["mean"].to_numpy(dtype=float)
            e = cond_df["sem"].fillna(0).to_numpy(dtype=float)

            ax.plot(
                x,
                y,
                marker="o",
                linewidth=2,
                color=CONDITION_COLORS[condition],
                label=CONDITION_LABELS.get(condition, condition)
            )           

            ax.fill_between(
                x,
                y - e,
                y + e,
                color=CONDITION_COLORS[condition],
                alpha=0.15
        )           

        ax.axhline(0, linestyle="--", linewidth=1, color="gray", alpha=0.7)
        ax.set_title(signal_map[signal], fontsize=18)
        ax.set_ylabel(r"$\Delta$ from baseline", fontsize=16)
        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=35, ha="right", fontsize=16)
        ax.spines["top"].set_visible(False)
        ax.spines["left"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["bottom"].set_linewidth(2)

        handles, labels = ax.get_legend_handles_labels()

    for ax in axes[n:]:
        ax.axis("off")

    if handles and labels:
        fig.legend(
            handles,
            labels,
            loc="upper center",
            ncol=3,
            frameon=False,
            bbox_to_anchor=(0.5, 1.02)
        )

    #fig.suptitle(suptitle, y=1.06)
    fig.tight_layout()

    out_path = PAPER_PHASE_DIR / filename
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", out_path)


def plot_difference_from_no_wind(df, signals, filename):
    """
    Plots wind_low - no_wind and wind_high - no_wind.
    This is often clearer than plotting all raw trajectories.
    Humanity may yet survive.
    """
    summary = summarize_phase_delta(df, signals)

    pivot = summary.pivot_table(
        index=["signal", "phase"],
        columns="condition",
        values="mean",
        observed=True
    ).reset_index()

    for cond in ["wind_low", "wind_high"]:
        if cond in pivot.columns and "no_wind" in pivot.columns:
            pivot[f"{cond}_minus_no_wind"] = pivot[cond] - pivot["no_wind"]

    available_signals = [s for s in signals if s in pivot["signal"].unique()]

    n = len(available_signals)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(12, 4.5 * nrows),
        sharex=True
    )

    axes = np.array(axes).reshape(-1)

    x = np.arange(len(PHASE_ORDER))
    x_labels = [PHASE_LABELS[p] for p in PHASE_ORDER]

    for ax, signal in zip(axes, available_signals):
        sig_df = (
            pivot[pivot["signal"] == signal]
            .set_index("phase")
            .reindex(PHASE_ORDER)
        )

        if "wind_low_minus_no_wind" in sig_df.columns:
            ax.plot(
                x,
                sig_df["wind_low_minus_no_wind"],
                marker="o",
                linewidth=2,
                color=CONDITION_COLORS["wind_low"],
                label="Low ventilation - No ventilation"
            )

        if "wind_high_minus_no_wind" in sig_df.columns:
            ax.plot(
                x,
                sig_df["wind_high_minus_no_wind"],
                marker="o",
                linewidth=2,
                color=CONDITION_COLORS["wind_high"],
                label="High ventilation - No ventilation"
            )

        ax.axhline(0, linestyle="--", linewidth=1, color="gray", alpha=0.7)
        ax.set_title(signal)
        ax.set_ylabel(r"$\Delta$ vs no ventilation")
        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=35, ha="right")
        ax.spines[["top", "right"]].set_visible(False)

    for ax in axes[n:]:
        ax.axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.5, 1.02)
    )

    fig.suptitle("Ventilation effect relative to no ventilation", y=1.06)
    fig.tight_layout()

    out_path = PAPER_PHASE_DIR / filename
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", out_path)


# ============================================================
# Run
# ============================================================

df_phase = load_data()

main_summary = summarize_phase_delta(df_phase, list(MAIN_PANEL_SIGNALS.keys()))
rh_summary = summarize_phase_delta(df_phase, list(RH_PANEL_SIGNALS.keys()))

plot_signal_panel(
    main_summary,
    MAIN_PANEL_SIGNALS,
    filename="paper_main_thermal_hygrometric_response.png",
    #suptitle="Thermal and hygrometric phase response"
)

plot_signal_panel(
    rh_summary,
    RH_PANEL_SIGNALS,
    filename="paper_rh_response_selected_locations.png",
    #suptitle="Relative humidity phase response at selected locations"
)

plot_difference_from_no_wind(
    df_phase,
    signals=list(MAIN_PANEL_SIGNALS.keys()),
    filename="paper_main_response_difference_vs_no_wind.png"
)

plot_difference_from_no_wind(
    df_phase,
    signals=list(RH_PANEL_SIGNALS.keys()),
    filename="paper_rh_response_difference_vs_no_wind.png"
)

print(f"\nDone. Paper-oriented phase plots saved in:\n{PAPER_PHASE_DIR}")

Saved: /home/eleonorab/repository/convective_cooling/reports/02_phase_features/paper_figures/paper_main_thermal_hygrometric_response.png
Saved: /home/eleonorab/repository/convective_cooling/reports/02_phase_features/paper_figures/paper_rh_response_selected_locations.png
Saved: /home/eleonorab/repository/convective_cooling/reports/02_phase_features/paper_figures/paper_main_response_difference_vs_no_wind.png
Saved: /home/eleonorab/repository/convective_cooling/reports/02_phase_features/paper_figures/paper_rh_response_difference_vs_no_wind.png

Done. Paper-oriented phase plots saved in:
/home/eleonorab/repository/convective_cooling/reports/02_phase_features/paper_figures
